# StringSense ABSA Labeling — Immutable Experiment Stage

This notebook generates review-aspect labels from the extracted, read-only JSON source. It never opens `archive_latest.zip` and never overwrites `data/*_latest.csv`. All outputs are created once under `output/runs/<run-id>/labeling/`.

## Run contract

Set `STRINGSENSE_NLP_RUN_ID` before launching Jupyter, or use `scripts/run_experiment.py`. Identical normalized review text is the split group, and every aspect row inherits that group split.

In [ ]:
import json
import os
import sys
from pathlib import Path

BASE_DIR = Path.cwd()
if not (BASE_DIR / 'data/archive_latest/badminton_strings_data.json').is_file():
    BASE_DIR = BASE_DIR / 'ml/nlp-workbench-latest'
if not (BASE_DIR / 'data/archive_latest/badminton_strings_data.json').is_file():
    raise FileNotFoundError('Cannot locate the canonical NLP workbench')
sys.path.insert(0, str(BASE_DIR / 'src'))

from stringsense_nlp.labeling import run_labeling

RUN_ID = os.environ.get('STRINGSENSE_NLP_RUN_ID')
if not RUN_ID:
    raise RuntimeError('Set STRINGSENSE_NLP_RUN_ID or use scripts/run_experiment.py')
RUN_ID

## Generate versioned labeling datasets

In [ ]:
labeling_result = run_labeling(RUN_ID, BASE_DIR)
labeling_result

## Verify the leakage gate and manifest

In [ ]:
with Path(labeling_result['manifest_path']).open('r', encoding='utf-8') as handle:
    labeling_manifest = json.load(handle)

assert labeling_manifest['leakage']['long']['review_cross_partition_count'] == 0
assert labeling_manifest['leakage']['long']['text_cross_partition_count'] == 0
assert labeling_manifest['leakage']['high_confidence']['review_cross_partition_count'] == 0
assert labeling_manifest['leakage']['high_confidence']['text_cross_partition_count'] == 0
labeling_manifest['dataset']

The pipeline notebook must now run with the same `STRINGSENSE_NLP_RUN_ID`. Promotion to the backend artifact is deliberately outside this notebook and requires human approval.